# Model Export Preparation
Export the final model to a SavedModel format and build a self-contained inference pipeline for use in the Streamlit dashboard.

In [1]:
import numpy as np
import cv2
import json
from pathlib import Path

import tensorflow as tf
from tensorflow import keras

2026-03-04 20:24:52.417327: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-04 20:24:53.205659: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-04 20:25:09.885310: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
PROJECT_ROOT = Path("../").resolve()
DATA_DIR = PROJECT_ROOT / "data"
ORGANIZED_DIR = DATA_DIR / "organized"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
EXPORT_DIR = PROJECT_ROOT / "app" / "model"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 256
CLASS_NAMES = ['AKIEC', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'VASC']

## 1. Load and Export Model

In [3]:
model_dir = MODELS_DIR / "final"

keras_path = model_dir / "best_model.keras"
h5_path = model_dir / "best_model.h5"

if keras_path.exists():
    model_path = keras_path
elif h5_path.exists():
    model_path = h5_path
else:
    raise FileNotFoundError("Best model not found (.keras or .h5)")

model = keras.models.load_model(str(model_path))

print(f"Model loaded from {model_path.name}. Layers: {len(model.layers)}")

I0000 00:00:1772655921.224198  569898 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13685 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Ti, pci bus id: 0000:01:00.0, compute capability: 8.9
2026-03-04 20:25:21.831412: E tensorflow/core/util/util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


Model loaded from best_model.h5. Layers: 344


In [4]:
print(f"Model loaded: {model.name}")
print(f"Input shape:  {model.input_shape}")
print(f"Output shape: {model.output_shape}")

Model loaded: EfficientNet_Baseline
Input shape:  (None, 256, 256, 3)
Output shape: (None, 8)


In [5]:
# Save as TensorFlow SavedModel (preferred for deployment)
model.export(str(EXPORT_DIR / "saved_model"))
print(f"Exported to: {EXPORT_DIR / 'saved_model'}")

INFO:tensorflow:Assets written to: pro/app/model/saved_model/assets


INFO:tensorflow:Assets written to: pro/app/model/saved_model/assets


Saved artifact at 'pro/app/model/saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 8), dtype=tf.float16, name=None)
Captures:
  130761312465344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130761299726352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130761299735152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130761299924016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130766996936608: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float16, name=None)
  130762286859184: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float16, name=None)
  130761153166000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130761153159664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130761153159488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  130761153164240: TensorSpec(shape=(), dtype=tf.resource, n

In [6]:
# Also save .keras for direct Streamlit loading
model.save(str(EXPORT_DIR / "model.keras"))
print("Keras model saved.")

Keras model saved.


## 2. Save Class Metadata

In [12]:
class_info = {
    "class_names": CLASS_NAMES,
    "img_size": IMG_SIZE,
    "descriptions": {
        "AKIEC":   "Actinic Keratosis — precancerous rough/scaly patch",
        "BCC":  "Basal Cell Carcinoma — most common skin cancer",
        "BKL":  "Benign Keratosis — non-cancerous skin growth",
        "DF":   "Dermatofibroma — benign fibrous nodule",
        "MEL":  "Melanoma — most dangerous skin cancer",
        "NV":   "Melanocytic Nevus — common mole (benign)",
        "VASC": "Vascular Lesion — blood vessel abnormality"
    },
    "clinical_risk": {
        "AKIEC":   "Precancerous",
        "BCC":  "Malignant",
        "BKL":  "Benign",
        "DF":   "Benign",
        "MEL":  "Malignant",
        "NV":   "Benign",
        "VASC": "Benign"
    }
}

with open(EXPORT_DIR / "class_info.json", "w") as f:
    json.dump(class_info, f, indent=2)

print("Class metadata saved.")

Class metadata saved.


## 3. Build and Test Inference Function
This is the exact function used inside the Streamlit app.

In [13]:
def preprocess_for_inference(img_bgr, target_size=(IMG_SIZE, IMG_SIZE)):
    """Resize, hair-remove, CLAHE — mirrors the training pipeline."""
    img = cv2.resize(img_bgr, target_size, interpolation=cv2.INTER_LINEAR)

    # Hair removal
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (17, 17))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, hair_mask = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)
    img = cv2.inpaint(img, hair_mask, 1, cv2.INPAINT_TELEA)

    # CLAHE
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab = cv2.merge([clahe.apply(l), a, b])
    img = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img_rgb

In [14]:
def predict(model, img_bgr):
    """Run full inference. Returns dict of class -> probability."""
    preprocessed = preprocess_for_inference(img_bgr)
    arr = tf.cast(preprocessed, tf.float32)[tf.newaxis, ...]
    probs = model(arr, training=False).numpy()[0]
    return {cls: float(p) for cls, p in zip(CLASS_NAMES, probs)}

In [15]:
# Quick smoke test
sample_path = next((ORGANIZED_DIR / "test" / "MEL").glob("*.jpg"))
sample_bgr = cv2.imread(str(sample_path))

result = predict(model, sample_bgr)
top_pred = max(result, key=result.get)

print(f"Sample: {sample_path.name}  (true label: MEL)")
print(f"Predicted: {top_pred} ({result[top_pred]:.2%})")
print("\nAll probabilities:")
for cls, prob in sorted(result.items(), key=lambda x: -x[1]):
    print(f"  {cls}: {prob:.4f}")

Sample: ISIC_0024367.jpg  (true label: MEL)
Predicted: MEL (84.18%)

All probabilities:
  MEL: 0.8418
  NV: 0.0853
  AKIEC: 0.0486
  BKL: 0.0171
  BCC: 0.0052
  DF: 0.0017
  VASC: 0.0000
